In [1]:
from ukrdc.database import Connection
from sqlalchemy.orm import sessionmaker

engine = Connection.get_engine_from_file(key="ukrdc_staging")

ukrdc3_sessionmaker = sessionmaker(
    autocommit=False, autoflush=False, bind=engine
)

ukrdc3 = ukrdc3_sessionmaker()

# Patients Treatment Statistics
The following shows how to generate some statistics demonstrating the propotion of patients on home therapies. These are calculated from June 2021 to June 2022 and show the breakdown of the different dialysis types as well as the treatment is occuring at home or in centre. 

The final histogram shows the mean number of times each in-centre dialysis patient recieves RRT. Statistics like this can be used to show the general compsition of the patients and the treatments they are receiving.  

In [2]:

from ukrdc_stats.calculators.dialysis import DialysisStatsCalculator
import datetime as dt

calculator = DialysisStatsCalculator(
    ukrdc3, "RNJ00", from_time=dt.datetime(2021, 12, 31), to_time=dt.datetime(2022, 12, 31)
)

dialysis_stats = calculator.extract_stats()



2023-01-13 00:00:00
2022-01-03 00:00:00
2022-01-11 00:00:00
2022-12-17 00:00:00
2022-01-04 00:00:00
2022-02-09 00:00:00
2021-11-29 00:00:00
2022-02-15 00:00:00
2022-04-19 00:00:00
2022-11-01 00:00:00
2021-12-21 00:00:00
2022-05-12 00:00:00
2022-01-13 00:00:00
2022-02-24 00:00:00
2022-10-20 00:00:00
2022-11-15 00:00:00
2021-11-06 00:00:00
2022-01-19 00:00:00
2022-07-18 00:00:00
2022-12-31 00:00:00
2022-03-10 00:00:00
2022-05-25 00:00:00
2023-01-03 00:00:00
2023-01-17 00:00:00
2022-06-22 00:00:00
2022-08-08 00:00:00
2022-08-25 00:00:00
2023-02-06 00:00:00
2022-05-17 00:00:00
2022-07-09 00:00:00
2023-03-24 00:00:00
2022-01-20 00:00:00
2022-04-20 00:00:00
2021-12-14 00:00:00
2021-11-09 00:00:00
2022-02-18 00:00:00
2022-11-18 00:00:00
2022-12-07 00:00:00
2023-02-07 00:00:00
2022-01-04 00:00:00
2022-04-06 00:00:00
2022-04-12 00:00:00
2022-07-11 00:00:00
2021-10-12 00:00:00
2021-10-20 00:00:00
2021-11-18 00:00:00
2022-01-06 00:00:00
2022-01-07 00:00:00
2022-04-12 00:00:00
2022-06-07 00:00:00


C:\Users\philip.main\Source\dashboard-stats\ukrdc_stats\calculators\dialysis.py:134: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  patient_cohort["qbl05"].replace(mappings, inplace=True)
C:\Users\philip.main\Source\dashboard-stats\ukrdc_stats\calculators\dialysis.py:134: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  patient_cohort["qbl05"].rep

In [3]:
print(list(dialysis_stats.units.keys()))

['RNJ00', '9RNJ00', 'RGCNH', 'RGCKH', 'RF4DG', '8CJ07']


In [4]:
from os import access
from ukrdc_stats.calculators.dialysis import DialysisStatsCalculator
import plotly.graph_objects as go
import plotly.express as px

import datetime as dt
from IPython.display import display

print(list(dialysis_stats.units.keys()))
subunit = "all"
#subunit = "RGCKH"

prev_patients = px.pie(
    names = dialysis_stats.all.prevalent_home_therapies.data.x,
    values = dialysis_stats.all.prevalent_home_therapies.data.y,
    title =  dialysis_stats.all.prevalent_home_therapies.metadata.title,
    hole=0.3,
)
prev_patients.show()

incident_patients = px.pie(
    names = dialysis_stats.all.incident_home_therapies.data.x,
    values = dialysis_stats.all.incident_home_therapies.data.y,
    title =  dialysis_stats.all.incident_home_therapies.metadata.title,
    hole=0.3,
)
incident_patients.show()



['RNJ00', '9RNJ00', 'RGCNH', 'RGCKH', 'RF4DG', '8CJ07']


# Patient Biomarker Statistics

**Note:** This section is currently not operational. Code related to biomarker stats was removed temporarily for a refactor and will be added back soon.

The following code shows some of the more granular statisics which can be produced using the data in the UKRDC. Here the eGFR results over a year can be plotted. These are harder to present in meaniful way but show how the data could be displayed to allow more microscopic insights to be generated. 

The bottom test shows all the eGFR in the given time period. From these results a breakdown of the CKD stage based on the most recent eGFR result of each patient. 

In [5]:
biomarker_results = calculator.extract_biomarkers()

AttributeError: 'DialysisStatsCalculator' object has no attribute 'extract_biomarkers'

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np


CKD_stage = px.pie(
    values=biomarker_results.CKD_stage.data.y,
    names=biomarker_results.CKD_stage.data.x,
    title=biomarker_results.CKD_stage.metadata.title,
    hole=0.3,
)
CKD_stage.show("png")


egfr_data = pd.DataFrame(
    list(
        zip(
            biomarker_results.prevalent_patients_egfr.x,
            [5 * i for i in range(len(biomarker_results.prevalent_patients_egfr.x))],
            biomarker_results.prevalent_patients_egfr.z,
        )
    ),
    columns=["date", "y", "eGFR"],
)

egfr_fig = px.scatter(
    egfr_data,
    x="date",
    y="y",
    color="eGFR",
    color_continuous_scale=px.colors.sequential.Inferno,
    range_color=[0, 30],
    #
    color_continuous_midpoint=15,
)

egfr_fig.update_layout({"plot_bgcolor": "rgba(0,0,0,0)"}, width=600, height=600)
egfr_fig.update_yaxes(visible=False)
egfr_fig.show("png")


print(min(egfr_data["eGFR"]))


# Time and Memory Usage
The next block of code will profile some of the time and memory usage of the stats calculated in this Demo. 

The main memory bottlenecks are the dataframes which store the query which is run on the ukrdc. These queries are also the main time bottleneck (note the extract_stats call now takes slightly longer to run than the initialisation call because I had to introduce some database queries for some of more complicated stats). 

# 